In [4]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import BernoulliRBM
from sklearn.metrics import accuracy_score, confusion_matrix

np.random.seed(12)

In [6]:
data = load_digits(n_class=5)
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train = X_train / 16.0
X_test = X_test / 16.0

In [7]:
rbm = BernoulliRBM(
    n_components=64,
    learning_rate=0.01,
    batch_size=10,
    n_iter=20,
    random_state=12
)

rbm.fit(X_train)

X_train_rbm = rbm.transform(X_train)
X_test_rbm = rbm.transform(X_test)

logistic_classifier = LogisticRegression(max_iter=1000)
logistic_classifier.fit(X_train_rbm, y_train)

y_pred_rbm = logistic_classifier.predict(X_test_rbm)

In [8]:
rbm1 = BernoulliRBM(
    n_components=100,
    learning_rate=0.06,
    batch_size=10,
    n_iter=25,
    random_state=12
)
rbm2 = BernoulliRBM(
    n_components=64,
    learning_rate=0.06,
    batch_size=10,
    n_iter=25,
    random_state=13
)

rbm1.fit(X_train)
X_train_hid1 = rbm1.transform(X_train)
X_test_hid1 = rbm1.transform(X_test)

rbm2.fit(X_train_hid1)
X_train_hid2 = rbm2.transform(X_train_hid1)
X_test_hid2 = rbm2.transform(X_test_hid1)

In [9]:
logistic_dbn = LogisticRegression(max_iter=1000)

logistic_dbn.fit(X_train_hid2, y_train)

y_pred_dbn = logistic_dbn.predict(X_test_hid2)

In [10]:
acc_rbm = accuracy_score(y_test, y_pred_rbm)
acc_dbn = accuracy_score(y_test, y_pred_dbn)

print(f"Single RBM + Logistic Regression Accuracy: {acc_rbm:.4f}")
print(f"Two-Layer DBN + Logistic Regression Accuracy: {acc_dbn:.4f}\n")

cm_dbn = confusion_matrix(y_test, y_pred_dbn)

print("DBN Confusion Matrix:")
print(cm_dbn)

Single RBM + Logistic Regression Accuracy: 0.9116
Two-Layer DBN + Logistic Regression Accuracy: 0.9337

DBN Confusion Matrix:
[[36  0  0  0  0]
 [ 0 27  7  3  0]
 [ 0  1 34  0  0]
 [ 0  0  1 36  0]
 [ 0  0  0  0 36]]


The two-layer DBN achieved higher accuracy than the single-RBM baseline, indicating that the additional RBM layer learned a more useful representation for classification. The confusion matrix shows that the largest error occurs when digit 1 is classified as digit 2.